In [1]:
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
MAX_PAGES = 5
DELAY_SECONDS = 2.0
TIMEOUT = 20

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BCIT-CourseMap/1.0; educational)"
}

COURSE_URLS = [
    "https://www.bcit.ca/courses/applied-data-management-for-analytics-babi-4000/",
    "https://www.bcit.ca/courses/computers-and-the-law-blaw-3600/",
    "https://www.bcit.ca/courses/dqm-with-python-babi-4005/",
]

In [3]:
def fetch_html(url):
    r = requests.get(url, timeout=TIMEOUT)  # requests follows redirects automatically
    if r.status_code != 200:
        print("Failed:", r.status_code, url)
        return None, None
    return r.text, r.url  # r.url is the final redirect target

In [3]:
def extract_after_label(page_text, label):
    """
    Finds the first occurrence of 'label' and returns the text after it,
    until the next line that looks like a new section header.
    """
    lines = [line.strip() for line in page_text.split("\n") if line.strip()]

    # find where the label appears
    start_index = None
    for i, line in enumerate(lines):
        if line.lower() == label.lower():
            start_index = i
            break

    if start_index is None:
        return None

    # collect lines after label until a "new section" begins
    collected = []
    for j in range(start_index + 1, len(lines)):
        line = lines[j]

        # stop if we hit another section header we care about
        if line.lower() in [
            "course overview",
            "prerequisite(s)",
            "credits",
            "learning outcomes",
            "related programs",
        ]:
            break

        collected.append(line)

    result = " ".join(collected).strip()
    return result if result else None